# 05 · RoBERTa classifier (Murarka et al., LOUHI 2021)

`hfl/chinese-roberta-wwm-ext` with the paper's recipe: lr 1e-5, 10 epochs (no early stopping),
batch 16, dropout 0.3, 512 tokens (`model_overrides` in `configs/finetune.yaml`); best dev epoch.
Deviations (Chinese model, balanced class weights, posts only): `docs/BASELINES.md` §3.

| | |
|---|---|
| Hardware | one GPU with 16 GB (T4) |
| Time (measured on a T4) | ~33 min per post-task run, 31 multiclass, 11 binary: ~5.4 GPU-h for 12 runs; Reddit sanity ~1 h |
| Needs | notebook 01; for the sanity check, access to Google Drive (the paper's data) |
| Writes | `outputs/finetune/<task>/roberta-wwm-ext/`, `outputs/tasks/reddit_posts/`, `outputs/finetune/reddit_posts/` |

The same work in one command: `python scripts/run_pipeline.py --no-limit --config configs/pipeline_roberta.yaml`.

> **Data use.** MATCHA is under a Data Use Agreement. These notebooks print counts and scores only, never posts; keep it that way if you add cells, and do not commit notebooks with outputs.

In [ ]:
import sys
from pathlib import Path

# make `matcha_bench` importable from a fresh clone (not needed after `pip install -e .`)
sys.path.insert(0, str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"))
from matcha_bench.notebook import check, setup, step

ROOT = setup()  # every step runs from the repository root
print(ROOT)

## 1. The MATCHA tasks

In [ ]:
MATCHA_TASKS = ["matcha_disorder_post", "matcha_disorder_post_masked",
                "matcha_user_binary_masked", "matcha_user_multiclass_masked"]
step("scripts/run_finetune.py", "--tasks", *MATCHA_TASKS, "--models", "roberta-wwm-ext")

## 2. Sanity check: `roberta-base` on the paper's Reddit posts

The authors' own train/val/test files (6 classes: adhd, anxiety, bipolar, depression, ptsd, none),
downloaded from the Google Drive folder linked in their repository; plain cross-entropy, one seed
(the paper reports one run). The released files hold 16,931 posts (13,727 / 1,716 / 1,488), 1.3%
fewer than the paper's 17,159.

In [ ]:
step("-m", "pip", "install", "gdown>=5.2")
step("scripts/download_reddit_data.py")
step("scripts/run_finetune.py", "--config", "configs/finetune_reddit.yaml", "--tasks", "reddit_posts",
     "--models", "roberta-base", "--seeds", "42")

In [ ]:
import json

meta = json.loads(Path("outputs/tasks/reddit_posts/meta.json").read_text(encoding="utf-8"))
check("Reddit split sizes", meta["sizes"], {"train": 13_727, "dev": 1_716, "test": 1_488})
m = json.loads(Path("outputs/finetune/reddit_posts/roberta-base/seed42/metrics.json").read_text(encoding="utf-8"))
check("Reddit test macro-F1", round(100 * m["test"]["f1"], 2), 85.24, tol=1.5)  # paper: 86

## 3. Scores

**Reported (test macro-F1, %, mean ± SD over seeds 42–44):** disorder_post 50.11 ± 0.59,
disorder_post_masked 41.76 ± 0.91, user_binary_masked 91.11 ± 0.59, user_multiclass_masked
50.65 ± 0.56. Reddit posts: 85.24 (paper 86).

In [ ]:
from matcha_bench.stats import collect_results, summarise

results = collect_results(Path("outputs/finetune"))
if results.empty:
    print("no finished runs yet")
else:
    print(summarise(results[results["model"].isin(["roberta-wwm-ext", "roberta-base"])]).to_string())

Next: `06_results.ipynb`.